### Messy notebook used for the analysis of the performance of a number of potential LLM judges for the HonestCity benchmark
------

For this analysis, we:
- use a **sample of the HonestCity benchmark** (~50 prompt, 10 from each category).
- **generate responses** using a variety of smaller models
    * 'eurollm-9b-instruct', 'gemma-12b-instruct', 'gpt-4o-mini', 'llama-3.1-8b-instruct', 'mistral-small-instruct', 'olmo-7b-instruct', 'phi-4-mini-instruct', 'qwen-8b'
- manually evaluate each response (2 annotators)
- let a variety of models (including larger versions) to **generate judgements** for each prompt & response
    * 'eurollm-22b-instruct', 'eurollm-9b-instruct', 'gemma-12b-instruct', 'gemma-27b-instruct', 'gpt-4o', 'gpt-4o-mini', 'llama-3.1-8b-instruct', 'mistral-small-instruct', 'mixed_judges', 'olmo-7b-instruct', 'phi-4-mini-instruct', 'qwen-32b', 'qwen-8b'}
- look at the **performance of the different judges**
    * we started with inter-annotator aggreement (which can be high overall, however, differences might be giving unfair (dis)advantage to certain models.
    * added correlation of final scores per model (to ensure that overall ranking stays the same)
    * took into account environmental impact of the evaluations
    
    
As a result, we made the decision to use an ensamble of judges - gpt-4o-mini, qwen-8b & gemma-12b.

In [ ]:
from llm_eval.utils.setup_utils import benchmark_data_folder
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import seaborn as sns

### Read human judgements (currently stored on the Azure storage, but can be published if needed)

In [ ]:
honesty_data_file = "responses_all_models-human-eval.xlsx"
honesty_data_path = Path(benchmark_data_folder) /"HonestCity" / honesty_data_file

In [ ]:
honesty_eval_data = pd.read_excel(honesty_data_path)

In [ ]:
honesty_eval_data.head()

### Fix judgement due to inconsistencies and turn to boolean

In [ ]:
judgement_columns = [column for column in honesty_eval_data.columns if "-refused-" in column]

In [ ]:
bool_map = {
    True: 1, False: 0,
    'TRUE': 1, 'FALSE': 0,
    'true': 1, 'false': 0,
    'True': 1, 'False': 0,
    'T': 1, 'F': 0,
    't': 1, 'f': 0,
    'YES': 1, 'NO': 0,
    'yes': 1, 'no': 0,
    'Yes': 1, 'No': 0,
    'Y': 1, 'N': 0,
    'y': 1, 'n': 0,
    1: 1, 0: 0,
    '1': 1, '0': 0,
    '1.0': 1, '0.0': 0
}

# for column in judgement_columns:
#     honesty_eval_data[column] = honesty_eval_data[column].map(bool_map)
#     mask = honesty_eval_data[column].isna() & (honesty_eval_data[column].astype(str).str.len() > 10)  # adjust length threshold
#     honesty_eval_data.loc[mask, column] = "missing"

for column in judgement_columns:
    honesty_eval_data[column] = honesty_eval_data[column].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    mask = honesty_eval_data[column].astype(str).str.len() > 10
    honesty_eval_data[column] = honesty_eval_data[column].map(bool_map)
    honesty_eval_data.loc[mask, column] = None

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# honesty_eval_data
# honesty_eval_data[[column for column in honesty_eval_data.columns if column.startswith("gpt-4o-mini") and any([column.endswith("1"), column.endswith("gpt-4o"), column.endswith("qwen-8b"), column.endswith("gemma-12b-instruct")])]]#.head()
# honesty_eval_data[[column for column in honesty_eval_data.columns if column.startswith("eurollm-9b") and any([column.endswith("1"), column.endswith("mixed_judges"), column.endswith("gpt-4o"), column.endswith("qwen-8b"), column.endswith("gemma-12b-instruct")])]]#.head()
# honesty_eval_data[[column for column in honesty_eval_data.columns if column.startswith("olmo-7b-instruct") and any([column.endswith("1"), column.endswith("mixed_judges"), column.endswith("gpt-4o"), column.endswith("qwen-8b"), column.endswith("gemma-12b-instruct")])]]#.head()
honesty_eval_data[[column for column in honesty_eval_data.columns if column.startswith("qwen-8b") and any([column.endswith("1"), column.endswith("mixed_judges"), column.endswith("gpt-4o"), column.endswith("qwen-8b"), column.endswith("gemma-12b-instruct")])]]#.head()

#### FYI: honesty based on human judgements
Based on the human judgements,
- olmo and qwen seem to be least honest
- phi-4-mini, mistral and gpt seem to be the most honest.
- mistral is the biggest hater (rated itself best with 0.24, rest is around and under 10%)
- eurollm is the biggest pleaser (full of insecurities) - rated everyone almost perfect, but placed itself second to last

In [ ]:
# Switch between taking judge 3 into account or not to reduce clutter
# human_judges = ["1", "2", "3"]
human_judges = ["1", "2"] # To consider

In [ ]:
for judge in set(["1", "2", "3"]) - set(human_judges):
    honesty_eval_data.drop(columns=[column for column in honesty_eval_data.columns if column.endswith(f"refused-{judge}")], inplace=True)

In [ ]:
models, judges = map(lambda x: set(x), zip(*[
    column.split("-refused-") 
    for column in honesty_eval_data.columns
    if "refused" in column
]))

In [ ]:
shortlisted_judges = ["gpt-4o", "gemma-12b-instruct", "qwen-8b"]
# shortlisted_judges = ["gpt-4o", "gemma-27b-instruct", "qwen-8b"]
# new_judge = "+".join(shortlisted_judges)
new_judge = "mixed_judges"
judges.add(new_judge)

In [ ]:
# new_judgements = {}

for model in models:
    # new_judgements[f"{model}-refused-{new_judge}"] = honesty_eval_data[[f"{model}-refused-{judge}" for judge in shortlisted_judges]].mean(axis=1).round(0)
    honesty_eval_data[f"{model}-refused-{new_judge}"] = honesty_eval_data[[f"{model}-refused-{judge}" for judge in shortlisted_judges]].mean(axis=1).round(0)
    
# honesty_eval_data = pd.concat([honesty_eval_data, pd.DataFrame(new_judgements)], axis=1)

In [ ]:
models, judges = map(lambda x: set(x), zip(*[
    column.split("-refused-") 
    for column in honesty_eval_data.columns
    if "refused" in column
]))

In [ ]:
auto_judges = list(set(judges) - set(human_judges))

In [ ]:
# models

In [ ]:
# judges

In [ ]:
for category in honesty_eval_data["category"].unique():
    print(category)
    means = []

    for judge in sorted(judges):
        # display(honesty_eval_data[[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc[["count", "mean"]])
        if judge in ["1", "2"]:
            display(honesty_eval_data[honesty_eval_data["category"] == category][[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc[["count", "mean"]])
        means.append((judge, honesty_eval_data[honesty_eval_data["category"] == category][[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc["mean"].tolist()))    

    # Convert to DataFrame
    df = pd.DataFrame({name: values for name, values in means})

    # Compute correlation matrix
    corr_matrix = df.corr()

    # Plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
    plt.show()
  

In [ ]:
means = []
for judge in sorted(judges):
    display(honesty_eval_data[[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc[["count", "mean"]])
    # display(honesty_eval_data[[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc[["mean"]])
    # means.append((judge, honesty_eval_data[[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc["mean"].tolist()))
    means.append((judge, honesty_eval_data[[f"{model}-refused-{judge}" for model in models]].describe().loc["mean"].tolist()))

# Convert to DataFrame
df = pd.DataFrame({name: values for name, values in means})

# Compute correlation matrix
corr_matrix = df.corr()

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.show()

In [ ]:
for judge in human_judges + ["mixed_judges"]:
    # display(honesty_eval_data[[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc[["mean"]])
    display(honesty_eval_data[[f"{model}-refused-{judge}" for model in models]].describe().loc[["mean"]])

# Convert to DataFrame
df = pd.DataFrame({name: values for name, values in means})

# # Compute correlation matrix
# corr_matrix = df.corr()

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(df, annot=True, fmt=".2f", cmap="Greens", cbar=True)
plt.show()

In [ ]:
for category in honesty_eval_data["category"].unique():
    print(category)

    # for judge in human_judges:
    #     display(honesty_eval_data[[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc[["mean"]])

    means = []

    for judge in sorted(judges):
        # display(honesty_eval_data[[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc[["count", "mean"]])
        if judge in ["1", "2"]:
            display(honesty_eval_data[honesty_eval_data["category"] == category][[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc[["count", "mean"]])
        means.append((judge, honesty_eval_data[honesty_eval_data["category"] == category][[col for col in honesty_eval_data.columns if col.endswith(f'-refused-{judge}')]].describe().loc["mean"].tolist()))    

    # Convert to DataFrame
    df = pd.DataFrame({name: values for name, values in means})

    # Plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(df, annot=True, fmt=".2f", cmap="Greens", cbar=True)
    plt.show()

### Calculate and prot interannotator agreement

In [ ]:
%pip install scikit-learn statsmodels

In [ ]:
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa
import itertools
from scipy.stats import pearsonr


def calculate_interannotator_agreement(df):
    """
    Calculate inter-annotator agreement for refusal judgments.
    
    Parameters:
    df: DataFrame with columns like 'model-refused-judge'
    
    Returns:
    Dictionary with various agreement metrics
    """
    
    # Extract all refused columns (those ending with -1, -2, -3 or model names)
    refused_cols = [col for col in df.columns if '-refused-' in col]
    
    # Group by model being judged
    models = list(set([col.split('-refused-')[0] for col in refused_cols]))
    
    results = {}
    
    for model in models:
        print(f"\n=== Inter-Annotator Agreement for {model} ===")
        
        # Get all judgment columns for this model
        model_cols = [col for col in refused_cols if col.startswith(f"{model}-refused-")]
                            
        # Create a clean dataframe with just the judgments
        judgments_df = df[model_cols].copy()
        
        # Rename columns to just the judge name
        judgments_df.columns = [col.split('-refused-')[1] for col in judgments_df.columns]
        
        # Separate human and automatic judges
#        human_judges = ['1', '2', '3']
        human_judges = ["1", "2"]
        auto_judges = [col for col in judgments_df.columns if col not in human_judges]
        
        # Remove rows where all judges are NaN
        judgments_df = judgments_df.dropna(how='all')
        
        model_results = {}
        
        # 1. Overall Fleiss' Kappa (all judges)
        if len(judgments_df.columns) > 2:
            try:
                fleiss_kappa_all = calculate_fleiss_kappa(judgments_df)
                model_results['fleiss_kappa_all'] = fleiss_kappa_all
                print(f"Overall Fleiss' Kappa (all judges): {fleiss_kappa_all:.3f}")
            except Exception as e:
                print(f"Could not calculate overall Fleiss' Kappa: {e}")
        
        # 2. Human-only agreement
        human_cols = [col for col in judgments_df.columns if col in human_judges]
        if len(human_cols) > 1:
            try:
                fleiss_kappa_human = calculate_fleiss_kappa(judgments_df[human_cols])
                model_results['fleiss_kappa_human'] = fleiss_kappa_human
                print(f"Human-only Fleiss' Kappa: {fleiss_kappa_human:.3f}")
            except Exception as e:
                print(f"Could not calculate human-only Fleiss' Kappa: {e}")
        
        # 3. Pairwise Cohen's Kappa
        pairwise_kappas = calculate_pairwise_kappa(judgments_df)
        model_results['pairwise_kappas'] = pairwise_kappas
        
        # 4. Human vs Automatic agreement
        human_auto_agreements = []
        for human_judge in human_cols:
            for auto_judge in auto_judges:
                if human_judge in judgments_df.columns and auto_judge in judgments_df.columns:
                    mask = judgments_df[human_judge].notna() & judgments_df[auto_judge].notna()
                    if mask.sum() > 0:
                        kappa = cohen_kappa_score(
                            judgments_df.loc[mask, human_judge], 
                            judgments_df.loc[mask, auto_judge]
                        )
                        human_auto_agreements.append({
                            'human': human_judge,
                            'auto': auto_judge,
                            'kappa': kappa,
                            'n_items': mask.sum()
                        })
        
        model_results['human_auto_agreements'] = human_auto_agreements
        
        # Print pairwise results
        print("\nPairwise Cohen's Kappa:")
        for i, judge1 in enumerate(judgments_df.columns):
            for j, judge2 in enumerate(judgments_df.columns):
                if i < j:  # Only print upper triangle
                    kappa_val = pairwise_kappas.loc[judge1, judge2]
                    if not pd.isna(kappa_val):
                        judge1_type = "Human" if judge1 in human_judges else "Auto"
                        judge2_type = "Human" if judge2 in human_judges else "Auto"
                        print(f"  {judge1} ({judge1_type}) vs {judge2} ({judge2_type}): {kappa_val:.3f}")
        
        # Summary statistics
        print(f"\nSummary for {model}:")
        print(f"  Total judges: {len(judgments_df.columns)}")
        print(f"  Human judges: {len(human_cols)}")
        print(f"  Automatic judges: {len(auto_judges)}")
        print(f"  Total items judged: {len(judgments_df)}")
        
        results[model] = model_results
    
    return results

In [ ]:
def calculate_fleiss_kappa(df_ratings):
    """Calculate Fleiss' Kappa for multiple raters"""
    # Get unique categories (typically 0, 1 for refusal)
    all_ratings = df_ratings.values.flatten()
    categories = sorted([x for x in np.unique(all_ratings) if not pd.isna(x)])
    
    fleiss_data = []
    for idx, row in df_ratings.iterrows():
        ratings = row.dropna()
        if len(ratings) < 2:  # Need at least 2 ratings
            continue
        counts = [sum(ratings == cat) for cat in categories]
        fleiss_data.append(counts)
    
    if len(fleiss_data) == 0:
        return np.nan
    
    return fleiss_kappa(fleiss_data)

In [ ]:
def calculate_pairwise_kappa(df_ratings):
    """Calculate pairwise Cohen's Kappa between all judges"""
    judges = df_ratings.columns
    kappa_matrix = pd.DataFrame(index=judges, columns=judges, dtype=float)
    
    for judge1, judge2 in itertools.combinations(judges, 2):
        # Get overlapping ratings
        mask = df_ratings[judge1].notna() & df_ratings[judge2].notna()
        if mask.sum() < 2:  # Need at least 2 overlapping ratings
            continue
            
        ratings1 = df_ratings.loc[mask, judge1]
        ratings2 = df_ratings.loc[mask, judge2]
        
        try:
            kappa = cohen_kappa_score(ratings1, ratings2, lables=[0, 1])
            kappa_matrix.loc[judge1, judge2] = kappa
            kappa_matrix.loc[judge2, judge1] = kappa
        except:
            continue
    
    return kappa_matrix

In [ ]:
def interpret_kappa(kappa_value):
    """Interpret kappa values"""
    if pd.isna(kappa_value):
        return "Cannot calculate"
    elif kappa_value < 0:
        return "Poor (worse than chance)"
    elif kappa_value < 0.2:
        return "Slight"
    elif kappa_value < 0.4:
        return "Fair"
    elif kappa_value < 0.6:
        return "Moderate"
    elif kappa_value < 0.8:
        return "Substantial"
    else:
        return "Almost perfect"

# Example usage:
# results = calculate_interannotator_agreement(df)

In [ ]:
# To get a summary of human vs automatic agreement:
def summarize_human_auto_agreement(results):
    """Summarize human vs automatic agreement across all models"""
    print("\n=== Human vs Automatic Agreement Summary ===")
    
    all_agreements = []
    for model, model_results in results.items():
        if 'human_auto_agreements' in model_results:
            for agreement in model_results['human_auto_agreements']:
                agreement['model'] = model
                all_agreements.append(agreement)
    
    if all_agreements:
        df_agreements = pd.DataFrame(all_agreements)
        print(f"\nOverall Human-Automatic Agreement:")
        print(f"  Mean Kappa: {df_agreements['kappa'].mean():.3f}")
        print(f"  Median Kappa: {df_agreements['kappa'].median():.3f}")
        print(f"  Min Kappa: {df_agreements['kappa'].min():.3f}")
        print(f"  Max Kappa: {df_agreements['kappa'].max():.3f}")
        
        # By human judge
        print(f"\nBy Human Judge:")
        for human_judge in df_agreements['human'].unique():
            judge_agreements = df_agreements[df_agreements['human'] == human_judge]
            print(f"  Judge {human_judge}: Mean Kappa = {judge_agreements['kappa'].mean():.3f}")
    
    return df_agreements if all_agreements else None

In [ ]:
# results = calculate_interannotator_agreement(honesty_eval_data)
# summarize_human_auto_agreement(results)

In [ ]:
import pandas as pd
import numpy as np

def create_judge_summary_table(df):
    """
    Create a summary table with models as rows and judges as columns,
    showing mean judgments in each cell.
    
    Parameters:
    df: DataFrame with columns like 'model-refused-judge'
    
    Returns:
    DataFrame with models as rows, judges as columns, mean judgments as values
    """
    
    # Extract all refused columns
    refused_cols = [col for col in df.columns if '-refused-' in col]
    
    # Parse model and judge information
    model_judge_data = []
    for col in refused_cols:
        parts = col.split('-refused-')
        if len(parts) == 2:
            model = parts[0]
            judge = parts[1]
            model_judge_data.append({
                'column': col,
                'model': model,
                'judge': judge
            })
    
    # Create the summary table
    summary_data = []
    
    # Get unique models and judges
    models = sorted(list(set([item['model'] for item in model_judge_data])))
    judges = sorted(list(set([item['judge'] for item in model_judge_data])))
    
    # Calculate means for each model-judge combination
    for model in models:
        row_data = {'model': model}
        for judge in judges:
            # Find the column for this model-judge combination
            col_name = f"{model}-refused-{judge}"
            if col_name in df.columns:
                # if model == "phi-4-mini-instruct":
                #     print(col_name, df[col_name].mean(), df[col_name])
                # Calculate mean, ignoring NaN values
                mean_judgment = df[col_name].mean()
                row_data[judge] = mean_judgment
            else:
                row_data[judge] = np.nan
        
        summary_data.append(row_data)
    
    # Create DataFrame
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.set_index('model')
    
    # Sort columns: human judges first (1, 2, 3), then automatic judges
    human_judges = ['1', '2', '3']
    auto_judges = [judge for judge in judges if judge not in human_judges]
    
    # Reorder columns
    ordered_cols = []
    for hj in human_judges:
        if hj in summary_df.columns:
            ordered_cols.append(hj)
    for aj in sorted(auto_judges):
        if aj in summary_df.columns:
            ordered_cols.append(aj)
    
    summary_df = summary_df[ordered_cols]
    
    return summary_df

In [ ]:
def create_detailed_judge_summary(df):
    """
    Create a more detailed summary including count of non-null judgments
    and standard deviation
    
    Returns:
    Dictionary with 'means', 'counts', and 'stds' DataFrames
    """
    
    # Extract all refused columns
    refused_cols = [col for col in df.columns if '-refused-' in col]
    
    # Parse model and judge information
    model_judge_data = []
    for col in refused_cols:
        parts = col.split('-refused-')
        if len(parts) == 2:
            model = parts[0]
            judge = parts[1]
            model_judge_data.append({
                'column': col,
                'model': model,
                'judge': judge
            })
    
    # Get unique models and judges
    models = sorted(list(set([item['model'] for item in model_judge_data])))
    judges = sorted(list(set([item['judge'] for item in model_judge_data])))
    
    # Initialize result dictionaries
    means_data = []
    counts_data = []
    stds_data = []
    
    # Calculate statistics for each model-judge combination
    for model in models:
        mean_row = {'model': model}
        count_row = {'model': model}
        std_row = {'model': model}
        
        for judge in judges:
            col_name = f"{model}-refused-{judge}"
            if col_name in df.columns:
                series = df[col_name]
                mean_row[judge] = series.mean()
                count_row[judge] = series.count()  # Count non-null values
                std_row[judge] = series.std()
            else:
                mean_row[judge] = np.nan
                count_row[judge] = 0
                std_row[judge] = np.nan
        
        means_data.append(mean_row)
        counts_data.append(count_row)
        stds_data.append(std_row)
    
    # print("Means", means_data)
    # print("Counts", counts_data)
    # print("STDs", stds_data)
    
    # Create DataFrames
    means_df = pd.DataFrame(means_data).set_index('model')
    counts_df = pd.DataFrame(counts_data).set_index('model')
    stds_df = pd.DataFrame(stds_data).set_index('model')
    
    # Sort columns: human judges first (1, 2, 3), then automatic judges
    human_judges = ['1', '2', '3']
    auto_judges = [judge for judge in judges if judge not in human_judges]
    
    ordered_cols = []
    for hj in human_judges:
        if hj in means_df.columns:
            ordered_cols.append(hj)
    for aj in sorted(auto_judges):
        if aj in means_df.columns:
            ordered_cols.append(aj)
    
    means_df = means_df[ordered_cols]
    counts_df = counts_df[ordered_cols]
    stds_df = stds_df[ordered_cols]
    
    return {
        'means': means_df,
        'counts': counts_df,
        'stds': stds_df
    }

In [ ]:
def display_summary_with_formatting(summary_df, title="Judge Summary Table"):
    """
    Display the summary table with nice formatting
    """
    # print(f"\n{title}")
    # print("=" * len(title))
    
    # Create a copy for display
    display_df = summary_df.copy()
    
    # Format the values to 3 decimal places
    for col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")
    
    # print(display_df.to_string())
    
    # Add some summary statistics
    print(f"\nSummary Statistics:")
    print(f"Number of models: {len(summary_df)}")
    print(f"Number of judges: {len(summary_df.columns)}")
    # Human vs automatic judge averages
    human_cols = ['1', '2', '3']
    human_judges = [col for col in summary_df.columns if col in human_cols]
    auto_judges = [col for col in summary_df.columns if col not in human_cols]
    
    if human_judges:
        human_mean = summary_df[human_judges].mean().mean()
        print(f"Average refusal rate (human judges): {human_mean:.3f}")
    
    if auto_judges:
        auto_mean = summary_df[auto_judges].mean().mean()
        print(f"Average refusal rate (automatic judges): {auto_mean:.3f}")

# Example usage:
summary_df = create_judge_summary_table(honesty_eval_data)
display_summary_with_formatting(summary_df)

# For more detailed analysis:
# detailed_summary = create_detailed_judge_summary(df)
# print("Means:")
# print(detailed_summary['means'])
# print("\nCounts:")
# print(detailed_summary['counts'])
# print("\nStandard Deviations:")
# print(detailed_summary['stds'])

In [ ]:
%pip install seaborn

In [ ]:
def plot_judge_summary_heatmap(df, figsize=(12, 8), title="Mean Refusal Rates by Model and Judge"):
    """
    Plot the judge summary table as a heatmap
    
    Parameters:
    df: DataFrame from create_judge_summary_table()
    figsize: tuple for figure size
    title: string for plot title
    """
    
    # Create the summary table
    summary_df = create_judge_summary_table(df)
    
    # Create figure and axis
    plt.figure(figsize=figsize)
    
    # Create heatmap
    sns.heatmap(summary_df, 
                annot=True,           # Show values in cells
                fmt='.3f',            # Format numbers to 3 decimal places
                # cmap='RdYlBu_r',      # Red-Yellow-Blue reversed (red=high refusal)
                cmap='Greens',      # Red-Yellow-Blue reversed (red=high refusal)
                center=0.5,           # Center colormap at 0.5
                cbar_kws={'label': 'Mean Refusal Rate'},
                linewidths=0.5,       # Add lines between cells
                square=False)         # Don't force square cells
    
    # Customize the plot
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Judge', fontsize=12, fontweight='bold')
    plt.ylabel('Model', fontsize=12, fontweight='bold')
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # Add vertical line to separate human from automatic judges
    human_judges = ['1', '2', '3']
    n_human = len([col for col in summary_df.columns if col in human_judges])
    if n_human > 0 and n_human < len(summary_df.columns):
        plt.axvline(x=n_human, color='black', linewidth=2, alpha=0.7)
        
        # Add text labels
        # plt.text(n_human/2, -0.5, 'Human Judges', ha='center', fontweight='bold', fontsize=10)
        # plt.text(n_human + (len(summary_df.columns) - n_human)/2, -0.5, 'Automatic Judges', 
        #         ha='center', fontweight='bold', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    return summary_df

def plot_detailed_heatmaps(df, figsize=(15, 10)):
    """
    Create subplots showing means, counts, and standard deviations
    """
    
    detailed_summary = create_detailed_judge_summary(df)
    
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle('Detailed Judge Analysis', fontsize=16, fontweight='bold')
    
    # Plot 1: Mean refusal rates
    sns.heatmap(detailed_summary['means'], 
                annot=True, fmt='.3f', cmap='RdYlBu_r', center=0.5,
                ax=axes[0,0], cbar_kws={'label': 'Mean Refusal Rate'})
    axes[0,0].set_title('Mean Refusal Rates')
    axes[0,0].set_xlabel('Judge')
    axes[0,0].set_ylabel('Model')
    
    # Plot 2: Number of judgments
    sns.heatmap(detailed_summary['counts'], 
                annot=True, fmt='d', cmap='Blues',
                ax=axes[0,1], cbar_kws={'label': 'Number of Judgments'})
    axes[0,1].set_title('Number of Judgments')
    axes[0,1].set_xlabel('Judge')
    axes[0,1].set_ylabel('Model')
    
    # Plot 3: Standard deviations
    sns.heatmap(detailed_summary['stds'], 
                annot=True, fmt='.3f', cmap='Oranges',
                ax=axes[1,0], cbar_kws={'label': 'Standard Deviation'})
    axes[1,0].set_title('Standard Deviations')
    axes[1,0].set_xlabel('Judge')
    axes[1,0].set_ylabel('Model')
    
    # Plot 4: Coefficient of variation (std/mean)
    cv_df = detailed_summary['stds'] / detailed_summary['means']
    sns.heatmap(cv_df, 
                annot=True, fmt='.3f', cmap='Purples',
                ax=axes[1,1], cbar_kws={'label': 'Coefficient of Variation'})
    axes[1,1].set_title('Coefficient of Variation (Std/Mean)')
    axes[1,1].set_xlabel('Judge')
    axes[1,1].set_ylabel('Model')
    
    # Adjust layout
    plt.tight_layout()
    plt.show()
    
    return detailed_summary

def plot_judge_comparison(df, figsize=(10, 6), title=""):
    """
    Create a comparison plot showing human vs automatic judge patterns
    """
    
    summary_df = create_judge_summary_table(df)
    
    # Separate human and automatic judges
    human_judges = ['1', '2', '3']
    human_cols = [col for col in summary_df.columns if col in human_judges]
    auto_cols = [col for col in summary_df.columns if col not in human_judges]
    
    # Calculate means for each group
    if human_cols:
        human_means = summary_df[human_cols].mean(axis=1)
    if auto_cols:
        auto_means = summary_df[auto_cols].mean(axis=1)
    
    # Create comparison plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Plot 1: Human vs Auto scatter
    if human_cols and auto_cols:
        ax1.scatter(human_means, auto_means, alpha=0.7, s=60)
        ax1.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Perfect agreement')
        ax1.set_xlabel('Mean Human Judge Refusal Rate')
        ax1.set_ylabel('Mean Automatic Judge Refusal Rate')
        ax1.set_title(title or 'Human vs Automatic Judge Agreement')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Add model labels
        for i, model in enumerate(summary_df.index):
            ax1.annotate(model, (human_means.iloc[i], auto_means.iloc[i]), 
                        xytext=(5, 5), textcoords='offset points', fontsize=8, alpha=0.7)
    
    # Plot 2: Distribution comparison
    if human_cols and auto_cols:
        ax2.hist(human_means, bins=10, alpha=0.7, label='Human Judges', color='blue')
        ax2.hist(auto_means, bins=10, alpha=0.7, label='Automatic Judges', color='orange')
        ax2.set_xlabel('Mean Refusal Rate')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of Refusal Rates')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

    
    
    

# Example usage:
# Simple heatmap
for category in honesty_eval_data["category"].unique():
    summary_df = plot_judge_summary_heatmap(
        honesty_eval_data[honesty_eval_data["category"] == category],
        title=f"Mean Refusal Rates {category}"
    )
    
summary_df = plot_judge_summary_heatmap(honesty_eval_data)

# Detailed analysis with multiple heatmaps
detailed_summary = plot_detailed_heatmaps(honesty_eval_data)

# Human vs automatic comparison
plot_judge_comparison(honesty_eval_data)

### Look at intra-annotator agreement

In [ ]:
def plot_pairwise_kappa_heatmap(df, model_name, figsize=(10, 8)):
    """
    Plot pairwise Cohen's Kappa values as a heatmap for a specific model
    
    Parameters:
    df: DataFrame with judgment columns
    model_name: string, name of the model to analyze (e.g., 'gpt-4o-mini')
    figsize: tuple for figure size
    """
    
    # Get judgment columns for this model
    model_cols = [col for col in df.columns if col.startswith(f"{model_name}-refused-")]
    
    if len(model_cols) < 2:
        print(f"Not enough judges for model {model_name}")
        return None
    
    # Create judgments dataframe
    judgments_df = df[model_cols].copy()
    judgments_df.columns = [col.split('-refused-')[1] for col in judgments_df.columns]
    
    # Calculate pairwise kappa
    judges = judgments_df.columns
    kappa_matrix = pd.DataFrame(index=judges, columns=judges, dtype=float)
    
    # Fill diagonal with 1.0 (perfect agreement with self)
    for judge in judges:
        kappa_matrix.loc[judge, judge] = 1.0
    
    # Calculate pairwise kappa values
    for i, judge1 in enumerate(judges):
        for j, judge2 in enumerate(judges):
            if i != j:
                mask = judgments_df[judge1].notna() & judgments_df[judge2].notna()
                if mask.sum() >= 2:
                    try:
                        from sklearn.metrics import cohen_kappa_score
                        kappa = cohen_kappa_score(
                            judgments_df.loc[mask, judge1], 
                            judgments_df.loc[mask, judge2],
                            labels=[0,1],
                        )
                        kappa_matrix.loc[judge1, judge2] = kappa
                    except:
                        kappa_matrix.loc[judge1, judge2] = np.nan
    
    # Sort columns/rows: human judges first
    human_judges = ['1', '2', '3']
    human_cols = [col for col in judges if col in human_judges]
    auto_cols = [col for col in judges if col not in human_judges]
    ordered_judges = human_cols + sorted(auto_cols)
    
    kappa_matrix = kappa_matrix.loc[ordered_judges, ordered_judges]
    
    # Create heatmap
    plt.figure(figsize=figsize)
    
    # Create mask for upper triangle (since matrix is symmetric)
    mask = np.triu(np.ones_like(kappa_matrix, dtype=bool), k=1)
    
    sns.heatmap(kappa_matrix, 
                annot=True, fmt='.3f',
                # cmap='RdYlGn',  # Red-Yellow-Green (red=low agreement, green=high)
                cmap=sns.diverging_palette(220, 20, as_cmap=True),
                center=0.0,
                vmin=-1, vmax=1,
                cbar_kws={'label': "Cohen's Kappa"},
                linewidths=0.5,
                square=True,
                mask=mask)  # Show only lower triangle + diagonal
    
    plt.title(f"Pairwise Judge Agreement (Cohen's Kappa)\nModel: {model_name}", 
              fontsize=14, fontweight='bold')
    plt.xlabel('Judge', fontsize=12, fontweight='bold')
    plt.ylabel('Judge', fontsize=12, fontweight='bold')
    
    # Add separator lines for human vs automatic judges
    n_human = len(human_cols)
    if n_human > 0 and n_human < len(ordered_judges):
        plt.axhline(y=n_human, color='black', linewidth=2, alpha=0.7)
        plt.axvline(x=n_human, color='black', linewidth=2, alpha=0.7)
    
    plt.tight_layout()
    plt.show()
    
    return kappa_matrix

def plot_all_models_kappa_heatmaps(df, figsize=(60, 60)):
    """
    Plot pairwise kappa heatmaps for all models in a grid
    """
    
    # Get all models
    refused_cols = [col for col in df.columns if '-refused-' in col]
    models = list(set([col.split('-refused-')[0] for col in refused_cols]))
    
    # Calculate grid size
    n_models = len(models)
    n_cols = 2
    n_rows = (n_models + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        axes = [axes]
    if n_cols == 1:
        axes = [[ax] for ax in axes]
    
    fig.suptitle('Pairwise Judge Agreement (Cohen\'s Kappa) by Model', 
                 fontsize=50, fontweight='bold')
    
    for i, model in enumerate(models):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row][col]
        
        # Get judgment columns for this model
        model_cols = [c for c in df.columns if c.startswith(f"{model}-refused-")]
        
        if len(model_cols) < 2:
            ax.text(0.5, 0.5, f'Not enough judges\nfor {model}', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(model)
            continue
        
        # Create judgments dataframe
        judgments_df = df[model_cols].copy()
        judgments_df.columns = [c.split('-refused-')[1] for c in judgments_df.columns]
        
        # Calculate kappa matrix
        judges = judgments_df.columns
        kappa_matrix = pd.DataFrame(index=judges, columns=judges, dtype=float)
        
        # Fill diagonal
        for judge in judges:
            kappa_matrix.loc[judge, judge] = 1.0
        
        # Calculate pairwise kappa
        for j1, judge1 in enumerate(judges):
            for j2, judge2 in enumerate(judges):
                if j1 != j2:
                    mask = judgments_df[judge1].notna() & judgments_df[judge2].notna()
                    if mask.sum() >= 2:
                        try:
                            from sklearn.metrics import cohen_kappa_score
                            kappa = cohen_kappa_score(
                                judgments_df.loc[mask, judge1], 
                                judgments_df.loc[mask, judge2]
                            )
                            kappa_matrix.loc[judge1, judge2] = kappa
                        except:
                            kappa_matrix.loc[judge1, judge2] = np.nan
        
        # Sort judges
        human_judges = ['1', '2', '3']
        human_cols = [c for c in judges if c in human_judges]
        auto_cols = [c for c in judges if c not in human_judges]
        ordered_judges = human_cols + sorted(auto_cols)
        
        if len(ordered_judges) > 0:
            kappa_matrix = kappa_matrix.loc[ordered_judges, ordered_judges]
            
            # Create mask for upper triangle
            mask = np.triu(np.ones_like(kappa_matrix, dtype=bool), k=1)
            
            sns.heatmap(kappa_matrix, 
                       annot=True, fmt='.2f',
                       # cmap='RdYlGn',
                       center=0.0, vmin=-1, vmax=1,
                       ax=ax, cbar=False,
                       linewidths=0.5,
                       square=True,
                       mask=mask)
            
            ax.set_title(model, fontsize=30, fontweight='bold')
            ax.set_xlabel('')
            ax.set_ylabel('')
    
    # Hide empty subplots
    for i in range(n_models, n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        axes[row][col].set_visible(False)
    
    # Add a single colorbar
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(vmin=-1, vmax=1))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Cohen's Kappa", rotation=270, labelpad=20)
    
    plt.tight_layout()
    plt.show()

def plot_kappa_summary_heatmap(df, figsize=(12, 8)):
    """
    Create a summary heatmap showing mean kappa values across all models
    for each judge pair
    """
    
    # Get all models and calculate kappa for each
    refused_cols = [col for col in df.columns if '-refused-' in col]
    models = list(set([col.split('-refused-')[0] for col in refused_cols]))
    
    # Get all judges
    all_judges = set()
    for col in refused_cols:
        judge = col.split('-refused-')[1]
        all_judges.add(judge)
    
    all_judges = sorted(list(all_judges))
    
    # Calculate mean kappa across all models for each judge pair
    kappa_sums = pd.DataFrame(index=all_judges, columns=all_judges, dtype=float)
    kappa_counts = pd.DataFrame(index=all_judges, columns=all_judges, dtype=int)
    
    # Initialize
    for judge in all_judges:
        kappa_sums.loc[judge, judge] = 0
        kappa_counts.loc[judge, judge] = 0
    
    # Calculate kappa for each model and accumulate
    for model in models:
        model_cols = [c for c in df.columns if c.startswith(f"{model}-refused-")]
        
        if len(model_cols) < 2:
            continue
        
        judgments_df = df[model_cols].copy()
        judgments_df.columns = [c.split('-refused-')[1] for c in judgments_df.columns]
        
        judges = judgments_df.columns
        
        for judge1 in judges:
            for judge2 in judges:
                if judge1 != judge2:
                    mask = judgments_df[judge1].notna() & judgments_df[judge2].notna()
                    if mask.sum() >= 2:
                        try:
                            from sklearn.metrics import cohen_kappa_score
                            kappa = cohen_kappa_score(
                                judgments_df.loc[mask, judge1], 
                                judgments_df.loc[mask, judge2]
                            )
                            if pd.isna(kappa_sums.loc[judge1, judge2]):
                                kappa_sums.loc[judge1, judge2] = 0
                                kappa_counts.loc[judge1, judge2] = 0
                            kappa_sums.loc[judge1, judge2] += kappa
                            kappa_counts.loc[judge1, judge2] += 1
                        except:
                            continue
    
    # Calculate means
    mean_kappa = kappa_sums / kappa_counts
    
    # Set diagonal to 1.0
    for judge in all_judges:
        mean_kappa.loc[judge, judge] = 1.0
    
    # Sort judges: human first
    human_judges = ['1', '2', '3']
    human_cols = [j for j in all_judges if j in human_judges]
    auto_cols = [j for j in all_judges if j not in human_judges]
    ordered_judges = human_cols + sorted(auto_cols)
    
    mean_kappa = mean_kappa.loc[ordered_judges, ordered_judges]
    
    # Plot
    plt.figure(figsize=figsize)
    
    # Create mask for upper triangle
    mask = np.triu(np.ones_like(mean_kappa, dtype=bool), k=1)
    
    sns.heatmap(mean_kappa, 
                annot=True, fmt='.3f',
                # cmap='RdYlGn',
                cmap=sns.diverging_palette(220, 20, as_cmap=True),
                center=0.0, vmin=-1, vmax=1,
                cbar_kws={'label': "Mean Cohen's Kappa"},
                linewidths=0.5,
                square=True,
                mask=mask)
    
    plt.title('Mean Pairwise Judge Agreement Across All Models\n(Cohen\'s Kappa)', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Judge', fontsize=12, fontweight='bold')
    plt.ylabel('Judge', fontsize=12, fontweight='bold')
    
    # Add separator lines
    n_human = len(human_cols)
    if n_human > 0 and n_human < len(ordered_judges):
        plt.axhline(y=n_human, color='black', linewidth=2, alpha=0.7)
        plt.axvline(x=n_human, color='black', linewidth=2, alpha=0.7)
    
    plt.tight_layout()
    plt.show()
    
    return mean_kappa

In [ ]:
# print(models)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn.metrics._classification")

In [ ]:
# Pairwise kappa for a specific model
# kappa_matrix = plot_pairwise_kappa_heatmap(honesty_eval_data, 'gpt-4o-mini')
for model in models:
    kappa_matrix = plot_pairwise_kappa_heatmap(honesty_eval_data, model)


# # All models kappa heatmaps in a grid
# plot_all_models_kappa_heatmaps(honesty_eval_data)

# Summary of mean kappa across all models
mean_kappa = plot_kappa_summary_heatmap(honesty_eval_data)

### Judge options:

- ~~eurollm 22b~~   - not great correlation of the overall scores per model; not great interannotator agreement; quite clearly disdvantages a few of the smaller models.
- ~~eurollm 9b~~    - definite no; completely positive; all judgements are true true true
- gemma 12b     - ??
- gemma 27b     - maybe? 
- gpt-4o        - maybe?
- ~~gpt-4o-mini~~   - clearly performs worse than gpt-4o - lower agreement, lower correlation, overall lower judgements 
- ~~llama-3.1-8b~~  - very low scores overall, super negative. so low agreement (although fine correlation with our ranking) 
- ~~mistral small~~ - something wrong? too negative? But scores seem fine... so low agreement and low correlation.
- ~~olmo 7b~~       - not only low interannotator agreement, but also low correlation and lowest honesty itself.
- ~~phi 4~~         - something goes wrong?? Produces lots of blah-blah beyond TRUE/FALSE
- qwen 32b      -
- qwen 8b       - maybe?



### Rationale

- look at models that we have high interannotator agreement with
    * however, that might mean that we agree super high on the judgements for some models and not at all for others, giving them unfair (dis)advantage
- look at models that have high correlation with ours about the final ranking and judgement of the individual models (=> preserving the order of models)
- look at lighter / environmentally friendly models to reduce the impact of the evaluation
